<a href="https://colab.research.google.com/github/pullz6/Large-Scale-Fraud-Detection-System/blob/main/Data_preprocessing_Model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 25.4 MB/s eta 0:00:00


In [3]:
!pip install -q kaggle

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from faker import Faker
import random

import os
from google.colab import drive
from kaggle.api.kaggle_api_extended import KaggleApi
import json

import pyspark
from pyspark.sql import Row
from datetime import datetime, timedelta

In [6]:
#Read creds
drive.mount('/content/drive')

#loading all the required datasets
creds_path = ('/content/drive/MyDrive/Projects/Creds/kaggle.json')

Mounted at /content/drive


In [7]:
with open(creds_path, 'r') as f:
    creds = json.load(f)

In [8]:
!mkdir -p ~/.kaggle

In [9]:
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(creds, f)

In [10]:
# Set permissions
!chmod 600 ~/.kaggle/kaggle.json

# Verify setup
!kaggle datasets list -s "ieee fraud"

# STEP 2: DOWNLOAD DATASET
# Download the dataset (this is a known mirror of the original competition data)
!kaggle datasets download -d kartik2112/fraud-detection

# Unzip the files (creates a 'fraud-detection' folder)
!unzip fraud-detection.zip -d fraud-detection

No datasets found
Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection
License(s): CC0-1.0
 55% 111M/202M [00:00<00:00, 1.16GB/s]
100% 202M/202M [00:00<00:00, 727MB/s] 
Archive:  fraud-detection.zip
  inflating: fraud-detection/fraudTest.csv  
  inflating: fraud-detection/fraudTrain.csv  


In [11]:
# List downloaded files
!ls -la fraud-detection/

total 489848
drwxr-xr-x 2 root root      4096 Jun 11 22:22 .
drwxr-xr-x 1 root root      4096 Jun 11 22:22 ..
-rw-r--r-- 1 root root 150354339 Aug  5  2020 fraudTest.csv
-rw-r--r-- 1 root root 351238196 Aug  5  2020 fraudTrain.csv


In [12]:
from pyspark.sql import SparkSession

In [13]:
spark = SparkSession.builder.appName("IEEE_Fraud_Detection").config("spark.driver.memory", "8g").config("spark.executor.memory", "8g").getOrCreate()

In [14]:
# Load the training and testing data
test = spark.read.csv(
    f"fraud-detection/fraudTest.csv",
    header=True,
    inferSchema=True
)

train = spark.read.csv(
    f"fraud-detection/fraudTrain.csv",
    header=True,
    inferSchema=True
)

In [15]:
# DataFrame approach
print(f"Training transactions: {train.count():,} rows")
train.printSchema()

# Show some statistics
train.describe().show()

Training transactions: 1,296,675 rows
root
 |-- _c0: integer (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: long (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: integer (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: integer (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: integer (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: integer (nullable = true)

+-------+-----------------+--------------------+-------------------+

In [16]:
test.describe().show()

+-------+-----------------+--------------------+-------------------+-------------+-----------------+-------+------+------+--------------------+-------+------+------------------+-----------------+------------------+------------------+------------------+--------------------+--------------------+-----------------+------------------+--------------------+
|summary|              _c0|              cc_num|           merchant|     category|              amt|  first|  last|gender|              street|   city| state|               zip|              lat|              long|          city_pop|               job|           trans_num|           unix_time|        merch_lat|        merch_long|            is_fraud|
+-------+-----------------+--------------------+-------------------+-------------+-----------------+-------+------+------+--------------------+-------+------+------------------+-----------------+------------------+------------------+------------------+--------------------+--------------------+

In [17]:
fake = Faker()

def generate_fake_fraud_data(num_records):
    data = []
    for _ in range(num_records):
        # Generate base transaction data
        cc_num = fake.credit_card_number()
        merchant = fake.company()
        category = random.choice(['entertainment', 'food_dining', 'gas_transport',
                                 'grocery_net', 'grocery_pos', 'health_fitness',
                                 'home', 'kids_pets', 'misc_net', 'misc_pos',
                                 'personal_care', 'shopping_net', 'shopping_pos',
                                 'travel'])
        amt = round(random.uniform(1, 1000), 2)

        # Generate personal information
        gender = random.choice(['M', 'F'])
        first = fake.first_name_male() if gender == 'M' else fake.first_name_female()
        last = fake.last_name()

        # Generate location data
        street = fake.street_address()
        city = fake.city()
        state = fake.state_abbr()
        zip_code = fake.zipcode()
        lat = float(fake.latitude())
        long = float(fake.longitude())
        city_pop = random.randint(1000, 1000000)

        # Generate employment data
        job = fake.job()

        # Transaction identifiers
        trans_num = fake.uuid4()
        unix_time = int(fake.date_time_this_year().timestamp())

        # Merchant location (slightly different from customer)
        merch_lat = lat + random.uniform(-0.1, 0.1)
        merch_long = long + random.uniform(-0.1, 0.1)

        # Fraud determination (5% base fraud rate)
        is_fraud = 0
        if random.random() < 0.05:
            is_fraud = 1
            # Make fraudulent transactions look different
            amt = round(random.uniform(500, 5000), 2)  # Higher amounts
            category = random.choice(['misc_net', 'shopping_net', 'travel'])  # Common fraud categories
            merch_lat = lat + random.uniform(-1, 1)  # More distant from customer
            merch_long = long + random.uniform(-1, 1)

        data.append(Row(
            _c0=len(data) + 1,  # Sequential ID
            cc_num=cc_num,
            merchant=merchant,
            category=category,
            amt=amt,
            first=first,
            last=last,
            gender=gender,
            street=street,
            city=city,
            state=state,
            zip=zip_code,
            lat=lat,
            long=long,
            city_pop=city_pop,
            job=job,
            trans_num=trans_num,
            unix_time=unix_time,
            merch_lat=merch_lat,
            merch_long=merch_long,
            is_fraud=is_fraud
        ))
    return data

# Generate 1000 fake records
fake_data = generate_fake_fraud_data(1000)

# Convert to Spark DataFrame
fake_df = spark.createDataFrame(fake_data)
fake_df.show(5)

+---+-------------------+--------------------+------------+------+-----+-------+------+--------------------+-------------+-----+-----+-----------+----------+--------+--------------------+--------------------+----------+-------------------+-------------------+--------+
|_c0|             cc_num|            merchant|    category|   amt|first|   last|gender|              street|         city|state|  zip|        lat|      long|city_pop|                 job|           trans_num| unix_time|          merch_lat|         merch_long|is_fraud|
+---+-------------------+--------------------+------------+------+-----+-------+------+--------------------+-------------+-----+-----+-----------+----------+--------+--------------------+--------------------+----------+-------------------+-------------------+--------+
|  1|   6011887297026373|Gordon, Fisher an...|shopping_net| 183.4|Kelly| Morris|     F|  2876 Meghan Branch|  Port Dennis|   MN|66464|-26.8432615| 65.441255|  995922|Sports developmen...|18ad70

In [18]:
combined_df = train.unionByName(fake_df, allowMissingColumns=True)

In [19]:
combined_df.describe().show()

+-------+-----------------+--------------------+----------------+-------------+------------------+-------+-------+-------+--------------------+---------+-------+------------------+-----------------+------------------+------------------+------------------+--------------------+--------------------+------------------+-------------------+--------------------+
|summary|              _c0|              cc_num|        merchant|     category|               amt|  first|   last| gender|              street|     city|  state|               zip|              lat|              long|          city_pop|               job|           trans_num|           unix_time|         merch_lat|         merch_long|            is_fraud|
+-------+-----------------+--------------------+----------------+-------------+------------------+-------+-------+-------+--------------------+---------+-------+------------------+-----------------+------------------+------------------+------------------+--------------------+--------

In [20]:
combined_df.printSchema()

root
 |-- _c0: long (nullable = true)
 |-- trans_date_trans_time: timestamp (nullable = true)
 |-- cc_num: string (nullable = true)
 |-- merchant: string (nullable = true)
 |-- category: string (nullable = true)
 |-- amt: double (nullable = true)
 |-- first: string (nullable = true)
 |-- last: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- street: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- city_pop: long (nullable = true)
 |-- job: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- trans_num: string (nullable = true)
 |-- unix_time: long (nullable = true)
 |-- merch_lat: double (nullable = true)
 |-- merch_long: double (nullable = true)
 |-- is_fraud: long (nullable = true)

